In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window

customers = spark.read.table("workspace.olist_silver.customers")
customer_orders = spark.read.table("workspace.olist_silver.orders_customer")

dim_customer = customer_orders.alias("co").select(col("co.order_id"),col("co.customer_id"), col("co.order_purchase_timestamp")).join(customers.alias("c"), on = "customer_id", how = "left")


dim_customer_partition = Window.partitionBy("customer_unique_id").orderBy(
    F.col("order_purchase_timestamp").desc_nulls_last(),
    F.col("order_id").desc_nulls_last()
)

dim_customer = (dim_customer.withColumn(
                                "rank",
                                F.row_number().over(dim_customer_partition)
                            )
                            .filter(F.col("rank") == 1)
                            .select(col("customer_unique_id").alias("customer_id"),
                                    "customer_city",
                                    "customer_state",
                                    "customer_zip_code_prefix"         
                            )
                )
dim_customer.write.mode("overwrite").saveAsTable("workspace.olist_gold.dim_customer")


In [0]:
%sql
describe detail workspace.olist_gold.dim_customer


In [0]:
items = spark.read.table("workspace.olist_silver.items")
display(items)

